In [1]:
# installing required packages
%pip install -q duckdb psutil pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
# checking system RAM and available disk space

import psutil
import shutil

ram_gb = psutil.virtual_memory().total / (1024**3)
free_disk_gb = shutil.disk_usage("/mnt/c").free / (1024**3)

print(f"System RAM: {ram_gb:.1f} GB")
print(f"Free disk space: {free_disk_gb:.1f} GB")

System RAM: 7.8 GB
Free disk space: 338.2 GB


In [3]:
# connecting to duckDB to get combined network logs file

import duckdb
import os

PARQUET_FILE = ("combined_uwf_dataset.parquet")

print(f"File exists: {os.path.exists(PARQUET_FILE)}")
print(f"File size: {os.path.getsize(PARQUET_FILE) / (1024**3):.2f} GB")


DB_FILE = (
    "uwf_gnn.duckdb"
)

con = duckdb.connect(DB_FILE)

con.execute("SET threads = 1")
con.execute("SET memory_limit = '4GB'")
con.execute("SET preserve_insertion_order = false")

File exists: True
File size: 1.42 GB


### Storing logs into a table in duckDB

In [30]:
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW raw_logs AS
SELECT *
FROM read_parquet('{PARQUET_FILE}')
""")

In [31]:
## dataset summary

dataset_summary = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(datetime) AS earliest_time,
    MAX(datetime) AS latest_time,
    approx_count_distinct(src_ip_zeek) AS source_ips,
    approx_count_distinct(dest_ip_zeek) AS destination_ips
FROM raw_logs
""").df()

dataset_summary

,total_rows,earliest_time,latest_time,source_ips,destination_ips
0,27111594,2021-12-17 07:00:45.213,2025-07-31 14:56:10.005,158,1008


In [6]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [7]:
con.execute("SELECT * FROM raw_logs LIMIT 5").df()

,resp_pkts,service,orig_ip_bytes,local_resp,missed_bytes,proto,duration,conn_state,dest_ip_zeek,orig_pkts,community_id,resp_ip_bytes,dest_port_zeek,orig_bytes,local_orig,datetime,history,resp_bytes,uid,src_port_zeek,ts,src_ip_zeek,label_tactic,source_period,label_technique,label_cve,label_binary,vlan
0,1689365,NaN,141906660,False,0,icmp,1.729902e+06,OTH,143.88.0.1,1689365,1:L8d7Q/9CNOBVbhfRe7yv3s8A24A=,141906660,0,94604440.0,False,2021-12-17 07:00:45.896,NaN,94604440.0,Caxllt2aEzXOEZnpii,8,1.639746e+09,143.88.255.50,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
1,0,NaN,462247452,False,0,icmp,1.730391e+06,OTH,143.88.255.10,4576716,1:IrLgQEMN3Jxld9DJo9XTcVvDsHU=,0,1,334099404.0,False,2021-12-17 07:00:47.630,NaN,0.0,CEx0rX1Yic8axI4sw1,3,1.639746e+09,143.88.255.1,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
2,0,NaN,22008480,False,0,icmp,1.730410e+06,OTH,ff02::1,229255,1:yBZ3xWqbIFuQFB44sXmyC45EtSs=,0,133,11004240.0,False,2021-12-17 07:00:45.879,NaN,0.0,Cpn5bT1QW9MPFXikn4,134,1.639746e+09,fe80::250:56ff:fe9e:da15,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
3,0,NaN,34970488,False,0,icmp,1.730411e+06,OTH,ff02::16,460138,1:ogw3gR/knTC3m00NBC6RR+vqjXc=,0,0,9202760.0,False,2021-12-17 07:00:45.888,NaN,0.0,Ceo6B73UmSZVe8vsD4,143,1.639746e+09,fe80::250:56ff:fe9e:ef90,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown
4,0,dhcp,66459795,False,0,udp,1.731504e+06,S0,255.255.255.255,202621,1:t9O1j0qj71O4wJM7gnaHtgmfev8=,0,67,60786407.0,False,2021-12-17 07:01:10.260,D,0.0,CTCPKc2faq2rLYJA4f,68,1.639746e+09,0.0.0.0,none,2021-12-12%20-%202021-12-19,unknown,unknown,unknown,unknown


In [8]:
## checking attack labels

label_summary = con.execute("""
SELECT
    label_binary,
    COUNT(*) AS rows
FROM raw_logs
GROUP BY label_binary
ORDER BY rows DESC
""").df()

label_summary

,label_binary,rows
0,unknown,18562468
1,False,4274703
2,True,4241370
3,Duplicate,33053


Many rows with "unknown" label_binary

In [9]:
## Setting duckDB memory usage

import os

TEMP_DIR = (
    "/mnt/c/SJSU/Spring 2026/DATA 298A/"
    "02 Sample Data/UWF_Data/duckdb_temp"
)

os.makedirs(TEMP_DIR, exist_ok=True)

con.execute("SET memory_limit = '5GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR}'")

In [10]:
## comparing binary labels with tactic labels

label_check = con.execute("""
SELECT
    label_binary,
    CASE
        WHEN label_tactic IS NULL THEN 'missing'
        WHEN LOWER(TRIM(label_tactic)) IN ('none', 'unknown', '')
            THEN 'no_attack_tactic'
        ELSE 'has_attack_tactic'
    END AS tactic_status,
    COUNT(*) AS rows
FROM raw_logs
GROUP BY label_binary, tactic_status
ORDER BY label_binary, tactic_status
""").df()

label_check

,label_binary,tactic_status,rows
0,Duplicate,has_attack_tactic,33053
1,False,no_attack_tactic,4274703
2,True,has_attack_tactic,4241370
3,unknown,has_attack_tactic,9280869
4,unknown,no_attack_tactic,9281599


Unknown label binary has both attack tactic as well as no attack tactic. This could mean that both attack and normal events are there.

In [11]:
## Available MITRE techniques

technique_summary = con.execute("""
SELECT
    CASE
        WHEN label_technique IS NULL
             OR LOWER(TRIM(label_technique)) IN ('none', 'unknown', '')
            THEN 'missing_technique'
        ELSE 'valid_technique'
    END AS technique_status,
    COUNT(*) AS rows,
    COUNT(DISTINCT label_technique) AS unique_values
FROM raw_logs
GROUP BY technique_status
""").df()

technique_summary

,technique_status,rows,unique_values
0,missing_technique,22837171,2
1,valid_technique,4274423,28


In [12]:
## number of examples for each MITRE

technique_counts = con.execute("""
SELECT
    label_technique,
    COUNT(*) AS rows
FROM raw_logs
WHERE label_technique LIKE 'T%'
GROUP BY label_technique
ORDER BY rows DESC
""").df()

technique_counts

,label_technique,rows
0,T1046,1779059
1,T1595,1197737
2,T1110,871188
3,T1587,275471
4,T1018,30774
5,T1592,21405
6,T1590,21208
7,T1016,17625
8,T1548,14902
9,T1078,6048


### Data Preparation

**Cleaning traget variable**

**Creating mode_labels view**

In [32]:
con.execute("""
CREATE OR REPLACE VIEW model_labels AS

WITH binary_labels AS (
    SELECT
        *,

        CASE
            WHEN label_binary = 'True' THEN 1
            WHEN label_binary = 'False' THEN 0

            WHEN label_binary = 'unknown'
                 AND LOWER(TRIM(label_tactic)) = 'none'
                THEN 0

            WHEN label_binary = 'unknown'
                 AND label_tactic IS NOT NULL
                 AND LOWER(TRIM(label_tactic))
                     NOT IN ('none', 'unknown', '')
                THEN 1

            ELSE NULL
        END AS binary_target

    FROM raw_logs
    WHERE label_binary != 'Duplicate'
),

clean_targets AS (
    SELECT
        *,

        CASE
            WHEN binary_target = 0 THEN 'none'
            WHEN binary_target = 1
                 AND label_tactic IS NOT NULL
                 AND LOWER(TRIM(label_tactic))
                     NOT IN ('none', 'unknown', '')
                THEN TRIM(label_tactic)
            ELSE NULL
        END AS tactic_target,

        CASE
            WHEN binary_target = 1
                 AND regexp_full_match(
                     TRIM(label_technique),
                     'T[0-9]{4}([.][0-9]{3})?'
                 )
                THEN TRIM(label_technique)
            ELSE NULL
        END AS mitre_target

    FROM binary_labels
)

SELECT
    *,

    CASE
        WHEN binary_target = 0 THEN 'none'
        WHEN mitre_target IS NOT NULL THEN mitre_target
        WHEN binary_target = 1 AND tactic_target IS NOT NULL
            THEN tactic_target || '_unspecified'
        ELSE NULL
    END AS technique_context

FROM clean_targets
""")

- binary_target: 0 or 1
- tactic_target: Discovery, Exfiltration, etc.
- mitre_target: only genuine codes such as T1046
- technique_context: genuine code or a fallback such as Discovery_unspecified

Model will first be trained on binary_target, then mitre_target and lastly tactic target.

In [14]:
## verifying cleaned targets 

target_summary = con.execute("""
SELECT
    binary_target,
    COUNT(*) AS total_rows,
    COUNT(tactic_target) AS rows_with_tactic,
    COUNT(mitre_target) AS rows_with_exact_mitre,
    COUNT(technique_context) AS rows_with_context
FROM model_labels
GROUP BY binary_target
ORDER BY binary_target
""").df()

target_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,binary_target,total_rows,rows_with_tactic,rows_with_exact_mitre,rows_with_context
0,0,13556302,13556302,0,13556302
1,1,13522239,13522239,4241370,13522239


In [15]:
con.execute("""
SELECT
    label_binary,
    label_tactic,
    label_technique,
    binary_target,
    tactic_target,
    mitre_target,
    technique_context
FROM model_labels
LIMIT 20
""").df()

,label_binary,label_tactic,label_technique,binary_target,tactic_target,mitre_target,technique_context
0,unknown,none,unknown,0,none,None,none
1,unknown,none,unknown,0,none,None,none
2,unknown,none,unknown,0,none,None,none
3,unknown,none,unknown,0,none,None,none
4,unknown,none,unknown,0,none,None,none
5,unknown,none,unknown,0,none,None,none
6,unknown,none,unknown,0,none,None,none
7,unknown,none,unknown,0,none,None,none
8,unknown,none,unknown,0,none,None,none
9,unknown,none,unknown,0,none,None,none


In [16]:
con.execute("""
SELECT
    label_binary,
    label_tactic,
    label_technique,
    binary_target,
    tactic_target,
    mitre_target,
    technique_context
FROM model_labels
WHERE binary_target = 1
  AND mitre_target IS NOT NULL
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,label_binary,label_tactic,label_technique,binary_target,tactic_target,mitre_target,technique_context
0,True,Defense Evasion,T1078,1,Defense Evasion,T1078,T1078
1,True,Defense Evasion,T1078,1,Defense Evasion,T1078,T1078
2,True,Defense Evasion,T1078,1,Defense Evasion,T1078,T1078
3,True,Defense Evasion,T1078,1,Defense Evasion,T1078,T1078
4,True,Defense Evasion,T1078,1,Defense Evasion,T1078,T1078


In [17]:
con.execute("""
SELECT
    label_binary,
    label_tactic,
    label_technique,
    binary_target,
    tactic_target,
    mitre_target,
    technique_context
FROM model_labels
WHERE binary_target = 1
  AND mitre_target IS NULL
LIMIT 5
""").df()

,label_binary,label_tactic,label_technique,binary_target,tactic_target,mitre_target,technique_context
0,unknown,Discovery,unknown,1,Discovery,None,Discovery_unspecified
1,unknown,Discovery,unknown,1,Discovery,None,Discovery_unspecified
2,unknown,Discovery,unknown,1,Discovery,None,Discovery_unspecified
3,unknown,Discovery,unknown,1,Discovery,None,Discovery_unspecified
4,unknown,Discovery,unknown,1,Discovery,None,Discovery_unspecified


**Examining the timeline**

In [18]:
year_summary = con.execute("""
SELECT
    YEAR(datetime) AS year,
    COUNT(*) AS total_rows,

    SUM(
        CASE WHEN binary_target = 0 THEN 1 ELSE 0 END
    ) AS normal_rows,

    SUM(
        CASE WHEN binary_target = 1 THEN 1 ELSE 0 END
    ) AS attack_rows,

    COUNT(mitre_target) AS rows_with_exact_mitre,
    COUNT(DISTINCT mitre_target) AS unique_mitre_techniques

FROM model_labels
GROUP BY year
ORDER BY year
""").df()

year_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,total_rows,normal_rows,attack_rows,rows_with_exact_mitre,unique_mitre_techniques
0,2021,3895689,3895689.0,0.0,0,0
1,2022,15364051,5736249.0,9627802.0,346933,22
2,2024,4800145,2413379.0,2386766.0,2386766,12
3,2025,3018656,1510985.0,1507671.0,1507671,6


Need to check MITRE codes in 2024 and 2025 appear oin 2022

In [19]:
techniques_by_year = con.execute("""
SELECT
    mitre_target,

    SUM(CASE WHEN YEAR(datetime) = 2022
             THEN 1 ELSE 0 END) AS rows_2022,

    SUM(CASE WHEN YEAR(datetime) = 2024
             THEN 1 ELSE 0 END) AS rows_2024,

    SUM(CASE WHEN YEAR(datetime) = 2025
             THEN 1 ELSE 0 END) AS rows_2025

FROM model_labels
WHERE mitre_target IS NOT NULL
GROUP BY mitre_target
ORDER BY rows_2022 DESC, rows_2024 DESC, rows_2025 DESC
""").df()

techniques_by_year

,mitre_target,rows_2022,rows_2024,rows_2025
0,T1587,275471.0,0.0,0.0
1,T1592,21405.0,0.0,0.0
2,T1590,21208.0,0.0,0.0
3,T1046,16819.0,1283978.0,478262.0
4,T1595,8587.0,163708.0,1025442.0
5,T1548,3061.0,8309.0,3532.0
6,T1589,292.0,0.0,0.0
7,T1203,18.0,0.0,0.0
8,T1071,14.0,3.0,0.0
9,T1210,11.0,29.0,326.0


There are some MITRE codes in 2024 and 2025 that are not present in 2022, which is being used for training

The data is split into train(2021 and 2022) - validation(2024) and test (2025). First will train a classifier to classify if event is attack or not using "binary_target". Then for MITRE prediction using "target_mitre", if we see unseen MITRE like the ones in 2024 and 2025. we will classify them as other. Next, we can predict tactic_target for the missing ones included. After the model is trained , the final selected model is then trained completely on 2021 + 2022 + 2024 data so that model is aware of the unseen technique now

2021–2022 → Initial training
2024      → Model selection
2021–2024 → Final retraining
2025      → Final untouched test

### Train - Validation - Test Split

Splitting based on year

**Creating split_logs view**

In [33]:
con.execute("""
CREATE OR REPLACE VIEW split_logs AS
SELECT
    *,

    CASE
        WHEN YEAR(datetime) IN (2021, 2022) THEN 'train'
        WHEN YEAR(datetime) = 2024 THEN 'validation'
        WHEN YEAR(datetime) = 2025 THEN 'test'
    END AS data_split

FROM model_labels
WHERE binary_target IS NOT NULL
  AND YEAR(datetime) IN (2021, 2022, 2024, 2025)
""")

In [21]:
split_summary = con.execute("""
SELECT
    data_split,
    COUNT(*) AS total_rows,

    SUM(
        CASE WHEN binary_target = 0 THEN 1 ELSE 0 END
    ) AS normal_rows,

    SUM(
        CASE WHEN binary_target = 1 THEN 1 ELSE 0 END
    ) AS attack_rows,

    MIN(datetime) AS earliest_time,
    MAX(datetime) AS latest_time

FROM split_logs
GROUP BY data_split
ORDER BY earliest_time
""").df()

split_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,data_split,total_rows,normal_rows,attack_rows,earliest_time,latest_time
0,train,19259740,9631938.0,9627802.0,2021-12-17 07:00:45.213000,2022-10-23 22:07:45.793000
1,validation,4800145,2413379.0,2386766.0,2024-02-28 04:00:37.805000,2024-12-09 23:59:51.394266
2,test,3018656,1510985.0,1507671.0,2025-05-23 00:00:00.895212,2025-07-31 14:56:10.005000


Target column is balanced . No future records are included in training period. 

In [22]:
## saving splits as separate views

con.execute("""
CREATE OR REPLACE VIEW train_logs AS
SELECT *
FROM split_logs
WHERE data_split = 'train'
""")

con.execute("""
CREATE OR REPLACE VIEW validation_logs AS
SELECT *
FROM split_logs
WHERE data_split = 'validation'
""")

con.execute("""
CREATE OR REPLACE VIEW test_logs AS
SELECT *
FROM split_logs
WHERE data_split = 'test'
""")

In [23]:
## verifying splits

con.execute("""
SELECT 'train' AS split, COUNT(*) AS rows FROM train_logs

UNION ALL

SELECT 'validation', COUNT(*) FROM validation_logs

UNION ALL

SELECT 'test', COUNT(*) FROM test_logs
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



,split,rows
0,train,19259740
1,validation,4800145
2,test,3018656


In [24]:
## disabling duckDB progress messages

con.execute("PRAGMA disable_progress_bar")

### Preparing Graph Data

Context window - 1 hour (to start with) : How much history the model examines.


A window that is too small might contain only one or two connections. The GNN would have very little surrounding behavior to learn from.
A window that is too large might mix normal and attack activity together.

Edge Features:

- number of connections;
- total and average bytes;
- total packets;
- average duration;
- protocol;
- service;
- destination port;
- connection state.
These describe how the two computers communicated.


Node features

Each computer can have features such as:
- number of outgoing connections;
- number of incoming connections;
- number of unique computers contacted;
- total bytes sent;
- total bytes received.
These describe the computer’s overall behavior during that window.

Targets

- binary_target
- tactic_target
- mitre_target
- technique_context

**checking whether 5-minute prediction interval batches**

How many new events arrive every 5 mins, how many source - dest edges they create, how often an edge contains both normal and attack events

In [25]:
five_minute_summary = con.execute("""
WITH edge_batches AS (
    SELECT
        TIME_BUCKET(
            INTERVAL '5 minutes',
            datetime
        ) AS batch_start,

        src_ip_zeek,
        dest_ip_zeek,

        COUNT(*) AS event_count,
        MIN(binary_target) AS minimum_label,
        MAX(binary_target) AS maximum_label

    FROM train_logs

    GROUP BY
        batch_start,
        src_ip_zeek,
        dest_ip_zeek
),

batch_summary AS (
    SELECT
        batch_start,
        COUNT(*) AS edge_count,
        SUM(event_count) AS raw_event_count,

        SUM(
            CASE
                WHEN minimum_label = 0
                 AND maximum_label = 1
                THEN 1 ELSE 0
            END
        ) AS mixed_edges

    FROM edge_batches
    GROUP BY batch_start
)

SELECT
    COUNT(*) AS prediction_steps,

    ROUND(AVG(raw_event_count), 1) AS average_events,
    MEDIAN(raw_event_count) AS median_events,
    MAX(raw_event_count) AS maximum_events,

    ROUND(AVG(edge_count), 1) AS average_edges,
    MEDIAN(edge_count) AS median_edges,
    MAX(edge_count) AS maximum_edges,

    SUM(mixed_edges) AS total_mixed_edges

FROM batch_summary
""").df()

five_minute_summary

,prediction_steps,average_events,median_events,maximum_events,average_edges,median_edges,maximum_edges,total_mixed_edges
0,9300,2070.9,935.0,1937408.0,16.2,15.0,257,0.0


- 935 events are generated
- 15 unique source-dst edges
- 0 mixed edges

Five-minute batches do not mix normal and attack records on the same IP pair.

**Checking for 1 hour context**

In [26]:
hourly_context_summary = con.execute("""
WITH hourly_edges AS (
    SELECT
        DATE_TRUNC('hour', datetime) AS window_start,
        src_ip_zeek,
        dest_ip_zeek,

        COUNT(*) AS event_count,
        MIN(binary_target) AS minimum_label,
        MAX(binary_target) AS maximum_label

    FROM train_logs

    GROUP BY
        window_start,
        src_ip_zeek,
        dest_ip_zeek
),

hourly_graphs AS (
    SELECT
        window_start,
        COUNT(*) AS edge_count,
        SUM(event_count) AS raw_event_count,

        SUM(
            CASE
                WHEN minimum_label = 0
                 AND maximum_label = 1
                THEN 1 ELSE 0
            END
        ) AS mixed_edges

    FROM hourly_edges
    GROUP BY window_start
)

SELECT
    COUNT(*) AS context_windows,

    ROUND(AVG(raw_event_count), 1) AS average_events,
    MEDIAN(raw_event_count) AS median_events,
    MAX(raw_event_count) AS maximum_events,

    ROUND(AVG(edge_count), 1) AS average_edges,
    MEDIAN(edge_count) AS median_edges,
    MAX(edge_count) AS maximum_edges,

    SUM(mixed_edges) AS total_mixed_edges

FROM hourly_graphs
""").df()

hourly_context_summary

,context_windows,average_events,median_events,maximum_events,average_edges,median_edges,maximum_edges,total_mixed_edges
0,899,21423.5,11047.0,4132096.0,48.1,54.0,257,0.0


- 1 hour context has abt 54 edges
- 0 mixed edges

**Creating 5-min edges - Summary of connections**

In [34]:
months = con.execute("""
SELECT DISTINCT
    YEAR(datetime) AS year,
    MONTH(datetime) AS month
FROM split_logs
ORDER BY year, month
""").fetchall()

months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(2021, 12),
 (2022, 1),
 (2022, 2),
 (2022, 8),
 (2022, 9),
 (2022, 10),
 (2024, 2),
 (2024, 3),
 (2024, 9),
 (2024, 10),
 (2024, 11),
 (2024, 12),
 (2025, 5),
 (2025, 6),
 (2025, 7)]

**Buiulding 5-min edges month by month**

In [35]:
con.execute("DROP TABLE IF EXISTS edges_5min")

for index, (year, month) in enumerate(months):

    print(f"Processing {year}-{month:02d}...")

    edge_query = f"""
    SELECT
        data_split,

        TIME_BUCKET(
            INTERVAL '5 minutes',
            datetime
        ) AS window_start,

        src_ip_zeek,
        dest_ip_zeek,

        COUNT(*) AS connection_count,

        SUM(CAST(COALESCE(duration, 0) AS DOUBLE))
            AS total_duration,

        MAX(COALESCE(duration, 0))
            AS maximum_duration,

        SUM(CAST(COALESCE(orig_bytes, 0) AS DOUBLE))
            AS total_orig_bytes,

        SUM(CAST(COALESCE(resp_bytes, 0) AS DOUBLE))
            AS total_resp_bytes,

        SUM(CAST(COALESCE(orig_pkts, 0) AS DOUBLE))
            AS total_orig_packets,

        SUM(CAST(COALESCE(resp_pkts, 0) AS DOUBLE))
            AS total_resp_packets,

        SUM(CASE WHEN proto = 'tcp' THEN 1 ELSE 0 END)
            AS tcp_count,

        SUM(CASE WHEN proto = 'udp' THEN 1 ELSE 0 END)
            AS udp_count,

        SUM(CASE WHEN proto = 'icmp' THEN 1 ELSE 0 END)
            AS icmp_count,

        MIN(binary_target) AS minimum_binary_target,
        MAX(binary_target) AS binary_target

    FROM split_logs

    WHERE YEAR(datetime) = {year}
      AND MONTH(datetime) = {month}

    GROUP BY
        data_split,
        window_start,
        src_ip_zeek,
        dest_ip_zeek
    """

    if index == 0:
        con.execute(
            f"CREATE TABLE edges_5min AS {edge_query}"
        )
    else:
        con.execute(
            f"INSERT INTO edges_5min {edge_query}"
        )

    con.execute("CHECKPOINT")
    gc.collect()

Processing 2021-12...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing 2022-01...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing 2022-02...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing 2025-06...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing 2025-07...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [36]:
## verifying the table

con.execute("""
SELECT
    data_split,
    COUNT(*) AS edge_records,
    COUNT(DISTINCT window_start) AS prediction_steps,

    SUM(
        CASE
            WHEN minimum_binary_target != binary_target
            THEN 1 ELSE 0
        END
    ) AS mixed_binary_edges

FROM edges_5min
GROUP BY data_split
ORDER BY data_split
""").df()

,data_split,edge_records,prediction_steps,mixed_binary_edges
0,test,232088,5326,0.0
1,train,150914,9300,0.0
2,validation,295052,10560,0.0


In [44]:
con.execute("SELECT * FROM edges_5min LIMIT 5").df()

,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target
0,train,2021-12-18 06:10:00,143.88.255.10,10.0.10.1,820,0.011508,0.000064,73632.0,0.0,1640.0,0.0,0.0,820.0,0.0,0,0
1,train,2021-12-18 01:00:00,143.88.255.10,10.0.10.1,832,0.010901,0.000033,74736.0,0.0,1664.0,0.0,0.0,832.0,0.0,0,0
2,train,2021-12-17 21:30:00,143.88.1.53,224.0.0.251,3,0.000000,0.000000,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0,0
3,train,2021-12-18 21:40:00,143.88.1.51,143.88.1.1,60,300.246807,5.004674,10752.0,2880.0,240.0,240.0,0.0,60.0,0.0,0,0
4,train,2021-12-18 14:55:00,143.88.1.51,143.88.1.1,62,310.243275,5.004693,11064.0,2976.0,248.0,248.0,0.0,62.0,0.0,0,0


In [45]:
con.execute("SELECT * FROM edges_5min ORDER BY data_split DESC LIMIT 5").df()

,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target
0,validation,2024-12-09 18:30:00,143.88.0.42,8.8.8.8,23,0.000184,0.000018,1928.0,0.0,46.0,0.0,0.0,23.0,0.0,0,0
1,validation,2024-12-09 18:55:00,143.88.6.10,143.88.6.1,14,0.015001,0.002224,2248.0,2248.0,56.0,56.0,0.0,14.0,0.0,0,0
2,validation,2024-12-09 09:10:00,143.88.11.15,143.88.11.1,3,0.005506,0.001982,129.0,129.0,3.0,3.0,0.0,3.0,0.0,0,0
3,validation,2024-12-09 01:25:00,143.88.8.11,143.88.8.1,13,0.013081,0.001594,2336.0,2336.0,52.0,52.0,0.0,13.0,0.0,0,0
4,validation,2024-12-09 08:40:00,143.88.7.11,143.88.7.1,15,8.484922,8.472130,2640.0,2640.0,60.0,60.0,0.0,15.0,0.0,0,0


In [46]:
con.execute("SELECT COUNT(*) FROM edges_5min WHERE data_split = 'train'").df()

,count_star()
0,150914


In [47]:
## Saving the datasets

EDGE_OUTPUT_DIR = (
    "gnn_edge_data"
)

os.makedirs(EDGE_OUTPUT_DIR, exist_ok=True)


In [48]:
for split_name in ["train", "validation", "test"]:

    output_file = os.path.join(
        EDGE_OUTPUT_DIR,
        f"{split_name}_edges_5min.parquet"
    )

    con.execute(f"""
        COPY (
            SELECT *
            FROM edges_5min
            WHERE data_split = '{split_name}'
        )
        TO '{output_file}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    print(f"Saved: {output_file}")

Saved: gnn_edge_data/train_edges_5min.parquet
Saved: gnn_edge_data/validation_edges_5min.parquet
Saved: gnn_edge_data/test_edges_5min.parquet


In [39]:
## verifying the files

for split_name in ["train", "validation", "test"]:

    file_path = os.path.join(
        EDGE_OUTPUT_DIR,
        f"{split_name}_edges_5min.parquet"
    )

    row_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{file_path}')
    """).fetchone()[0]

    size_mb = os.path.getsize(file_path) / (1024**2)

    print(
        f"{split_name}: "
        f"{row_count:,} edges, "
        f"{size_mb:.1f} MB"
    )

train: 150,914 edges, 1.9 MB
validation: 295,052 edges, 3.9 MB
test: 232,088 edges, 2.4 MB


In [41]:
## checking target label balance after aggregation 

con.execute("""
SELECT
    data_split,
    binary_target,
    COUNT(*) AS edge_records,

    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (
            PARTITION BY data_split
        ),
        2
    ) AS percentage

FROM edges_5min

GROUP BY
    data_split,
    binary_target

ORDER BY
    data_split,
    binary_target
""").df()

,data_split,binary_target,edge_records,percentage
0,test,0,231130,99.59
1,test,1,958,0.41
2,train,0,150135,99.48
3,train,1,779,0.52
4,validation,0,242139,82.07
5,validation,1,52913,17.93


In [42]:
## checking whether 779 training edges occur across many time windows and IP pairs or few

con.execute("""
SELECT
    data_split,

    COUNT(*)
        FILTER (WHERE binary_target = 1)
        AS attack_edges,

    COUNT(DISTINCT window_start)
        FILTER (WHERE binary_target = 1)
        AS attack_windows,

    COUNT(
        DISTINCT (
            src_ip_zeek,
            dest_ip_zeek
        )
    )
        FILTER (WHERE binary_target = 1)
        AS attack_ip_pairs

FROM edges_5min

GROUP BY data_split
ORDER BY data_split
""").df()

,data_split,attack_edges,attack_windows,attack_ip_pairs
0,test,958,439,310
1,train,779,480,299
2,validation,52913,4896,632


In [49]:
con.execute("DROP TABLE IF EXISTS edge_targets_5min")

for index, (year, month) in enumerate(months):

    print(f"Processing targets for {year}-{month:02d}...")

    target_query = f"""
    SELECT
        data_split,

        TIME_BUCKET(
            INTERVAL '5 minutes',
            datetime
        ) AS window_start,

        src_ip_zeek,
        dest_ip_zeek,

        MAX(binary_target) AS binary_target,

        MODE(tactic_target)
            FILTER (
                WHERE binary_target = 1
                  AND tactic_target IS NOT NULL
                  AND tactic_target != 'none'
            )
            AS tactic_target,

        MODE(mitre_target)
            FILTER (
                WHERE binary_target = 1
                  AND mitre_target IS NOT NULL
            )
            AS mitre_target,

        COUNT(DISTINCT tactic_target)
            FILTER (
                WHERE binary_target = 1
                  AND tactic_target IS NOT NULL
                  AND tactic_target != 'none'
            )
            AS tactic_count,

        COUNT(DISTINCT mitre_target)
            FILTER (
                WHERE binary_target = 1
                  AND mitre_target IS NOT NULL
            )
            AS mitre_count

    FROM split_logs

    WHERE YEAR(datetime) = {year}
      AND MONTH(datetime) = {month}

    GROUP BY
        data_split,
        window_start,
        src_ip_zeek,
        dest_ip_zeek
    """

    if index == 0:
        con.execute(
            f"CREATE TABLE edge_targets_5min AS {target_query}"
        )
    else:
        con.execute(
            f"INSERT INTO edge_targets_5min {target_query}"
        )

    con.execute("CHECKPOINT")

Processing targets for 2021-12...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing targets for 2022-01...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing targets for 2022-02...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing targets for 2025-06...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing targets for 2025-07...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [50]:
con.execute("""
SELECT
    data_split,
    COUNT(*) AS target_rows,

    COUNT(tactic_target)
        AS rows_with_tactic,

    COUNT(mitre_target)
        AS rows_with_exact_mitre,

    SUM(
        CASE WHEN tactic_count > 1
             THEN 1 ELSE 0 END
    ) AS multiple_tactic_edges,

    SUM(
        CASE WHEN mitre_count > 1
             THEN 1 ELSE 0 END
    ) AS multiple_mitre_edges

FROM edge_targets_5min

GROUP BY data_split
ORDER BY data_split
""").df()

,data_split,target_rows,rows_with_tactic,rows_with_exact_mitre,multiple_tactic_edges,multiple_mitre_edges
0,test,232088,958,958,0.0,0.0
1,train,150914,779,439,23.0,321.0
2,validation,295052,52913,52913,1437.0,1441.0


- Binary prediction remains single-label: Normal or Attack.
- Tactic prediction can return one or more tactics.
- MITRE prediction can return one or more techniques.
- RAG can retrieve evidence for every predicted technique.

MITRE classification is going to be multi label classificatio. Because after summarization, each edge has more than one MITRE code.

In [51]:
con.execute("DROP TABLE IF EXISTS edge_target_lists_5min")

for index, (year, month) in enumerate(months):

    print(f"Processing target lists for {year}-{month:02d}...")

    target_query = f"""
    SELECT
        data_split,

        TIME_BUCKET(
            INTERVAL '5 minutes',
            datetime
        ) AS window_start,

        src_ip_zeek,
        dest_ip_zeek,

        MAX(binary_target) AS binary_target,

        LIST(DISTINCT tactic_target ORDER BY tactic_target)
            FILTER (
                WHERE binary_target = 1
                  AND tactic_target IS NOT NULL
                  AND tactic_target != 'none'
            )
            AS tactic_targets,

        LIST(DISTINCT mitre_target ORDER BY mitre_target)
            FILTER (
                WHERE binary_target = 1
                  AND mitre_target IS NOT NULL
            )
            AS mitre_targets

    FROM split_logs

    WHERE YEAR(datetime) = {year}
      AND MONTH(datetime) = {month}

    GROUP BY
        data_split,
        window_start,
        src_ip_zeek,
        dest_ip_zeek
    """

    if index == 0:
        con.execute(
            f"""
            CREATE TABLE edge_target_lists_5min
            AS {target_query}
            """
        )
    else:
        con.execute(
            f"""
            INSERT INTO edge_target_lists_5min
            {target_query}
            """
        )

    con.execute("CHECKPOINT")

Processing target lists for 2021-12...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing target lists for 2022-01...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing target lists for 2022-02...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing target lists for 2025-06...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing target lists for 2025-07...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [52]:
con.execute("""
SELECT
    data_split,
    window_start,
    src_ip_zeek,
    dest_ip_zeek,
    tactic_targets,
    mitre_targets
FROM edge_target_lists_5min
WHERE LEN(mitre_targets) > 1
LIMIT 10
""").df()

,data_split,window_start,src_ip_zeek,dest_ip_zeek,tactic_targets,mitre_targets
0,train,2022-08-31 17:50:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
1,train,2022-08-31 09:50:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
2,train,2022-08-31 14:50:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
3,train,2022-08-31 10:20:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
4,train,2022-08-31 07:30:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
5,train,2022-08-31 07:20:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
6,train,2022-08-31 03:50:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
7,train,2022-08-31 07:50:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
8,train,2022-08-31 21:05:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"
9,train,2022-08-31 18:40:00,143.88.10.11,143.88.255.10,[Reconnaissance],"[T1590, T1592]"


The same IP pair during one five-minute interval belongs to: T1590 and T1592

In [53]:
## Joining features and target

con.execute("""
CREATE OR REPLACE TABLE edges_5min_enriched AS

SELECT
    e.*,
    t.tactic_targets,
    t.mitre_targets

FROM edges_5min AS e

INNER JOIN edge_target_lists_5min AS t
    ON e.data_split = t.data_split
   AND e.window_start = t.window_start
   AND e.src_ip_zeek = t.src_ip_zeek
   AND e.dest_ip_zeek = t.dest_ip_zeek
""")

In [54]:
## verifying 

con.execute("""
SELECT
    data_split,
    COUNT(*) AS total_edges,

    SUM(
        CASE WHEN binary_target = 1
             THEN 1 ELSE 0 END
    ) AS attack_edges,

    COUNT(tactic_targets)
        AS edges_with_tactics,

    COUNT(mitre_targets)
        AS edges_with_mitre,

    SUM(
        CASE WHEN LEN(tactic_targets) > 1
             THEN 1 ELSE 0 END
    ) AS multi_tactic_edges,

    SUM(
        CASE WHEN LEN(mitre_targets) > 1
             THEN 1 ELSE 0 END
    ) AS multi_mitre_edges

FROM edges_5min_enriched

GROUP BY data_split
ORDER BY data_split
""").df()

,data_split,total_edges,attack_edges,edges_with_tactics,edges_with_mitre,multi_tactic_edges,multi_mitre_edges
0,test,232088,958.0,958,958,0.0,0.0
1,train,150914,779.0,779,439,23.0,321.0
2,validation,295052,52913.0,52913,52913,1437.0,1441.0


In [55]:
## verifying 5 min alignment

con.execute("""
SELECT COUNT(*) AS incorrectly_aligned_rows
FROM edges_5min_enriched
WHERE MINUTE(window_start) % 5 != 0
   OR SECOND(window_start) != 0
""").df()

,incorrectly_aligned_rows
0,0


In [56]:
ENRICHED_DIR = os.path.join(
    EDGE_OUTPUT_DIR,
    "enriched"
)

os.makedirs(ENRICHED_DIR, exist_ok=True)

In [57]:
for split_name in ["train", "validation", "test"]:

    output_file = os.path.join(
        ENRICHED_DIR,
        f"{split_name}_edges_5min_enriched.parquet"
    )

    con.execute(f"""
        COPY (
            SELECT *
            FROM edges_5min_enriched
            WHERE data_split = '{split_name}'
        )
        TO '{output_file}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    print(f"Saved: {output_file}")

Saved: gnn_edge_data/enriched/train_edges_5min_enriched.parquet
Saved: gnn_edge_data/enriched/validation_edges_5min_enriched.parquet
Saved: gnn_edge_data/enriched/test_edges_5min_enriched.parquet
